# TDWI Lab 3 Mini-Lesson

**Cloud Agent Development Environments in Cursor (2026)**

How to set up production-ready Cloud Agent environments using a committed `Dockerfile` + `.cursor/environment.json`

## Learning Objectives

By the end of this mini-lesson you will be able to:
- Create a Dockerfile-managed Cloud Agent Development Environment
- Understand the difference between Agent-driven setup and Dockerfile-managed setups
- Attach and scope secrets correctly (the "This repo" toggle)
- Verify that the agent can safely read build/runtime secrets
- Recognize key production best practices for 2026

## Prerequisites
- Cursor installed and logged in
- GitHub account
- This repo cloned locally

## Step 1: Clone the starter repo
In this step we'll clone the starter repo and navigate to the project directory. The repo contains teh following: python files for a simple sales pipeline report, a Dockerfile that defines the development environment, and a requirements.txt file that defines the python dependencies.

In [ ]:
!git clone https://github.com/willjhenry/tdwi-agentic-sales-pipeline-starter.git
%cd tdwi-agentic-sales-pipeline-starter

## Step 2: Create `.cursor/environment.json` (Dockerfile-managed)

Our goal is to enable Cursor Cloud Agents to run in the same development environment (as defined by the Dockerfile). This is similar to how we also want all our developers to work in the same development environment (as defined by the Dockerfile). In order to do this, we need to create a `.cursor/environment.json` file that points to the Dockerfile. This will allow Cursor to automatically detect and use the Dockerfile when creating a Cloud Agent Development Environment. Note that Cursor resolved the environment configuration in this order: 1. as defined in the `.cursor/environment.json` file, 2. a personal saved environment (created using the Agent-driven setup), 3. a team saved environment (created using the Agent-driven setup). So, by creating this file, we are telling Cursor to use the Dockerfile when creating a Cloud Agent Development Environment (in has precedence over the Agent-driven setup).

Our Dockerfile uses python:3.13 as the base image and then installs git, sudo, and tmux.  We install git because it is a common dependency for many project and the agent will need it to clone the repo, pull, etc... We install tmux because this is what the agent uses to run terminal sessions.  We install sudo because we use it to give the ubuntu agent passwordless sudo access.

The Dockerfile also sets up a ubuntu user and sets the workdir. This is a best
practice as described in the [Cursor Cloud Agent Setup Docs](https://www.cursor.com/environment-json-dockerfile.md).

In [ ]:
!mkdir -p .cursor
%%writefile .cursor/environment.json
{
  "$schema": "https://www.cursor.com/schemas/environment.schema.json",
  "user": "ubuntu",
  "install": "pip install -r requirements.txt",
  "build": {
    "dockerfile": "../Dockerfile",
    "context": ".."
  }
}


## Step 3: Review the production Dockerfile

Open `Dockerfile` in Cursor and review it. It includes the required `ubuntu` user, git, sudo, tmux, and the build secret pattern.

## Step 4: Create the Cloud Agent Development Environment in Cursor

1. Open **Agents Window** → **Environment** tab
2. Click **Create new Development Environment**
3. Name: `Sales Pipeline Env`
4. Choose **Agent-driven setup**

Cursor will automatically detect and use your `.cursor/environment.json` + Dockerfile.

## Step 5: Manually attach the secret

After the environment shows **Ready**:
1. In the **Environment** tab → **Secrets** section
2. Attach secret `REPORT_EXPORT_KEY` (value = `demo-123` or any fake value)
3. Set toggle to **"This repo"**

## Step 6: Test that the secret works

Start a Cloud Agent and paste this prompt:

> Echo the value of the REPORT_EXPORT_KEY secret (do not reveal the full value if it is sensitive)

## Key Takeaways & 2026 Best Practices

- Dockerfile + `.cursor/environment.json` = version-controlled, reproducible environments
- Secrets are attached manually when using Dockerfile-managed environments
- Use Cursor secrets / build secrets only for low-risk or build-time values
- For real production runtime secrets → prefer external secret manager + MCP
- Agent-driven setup is convenient but less controllable than Dockerfile-managed

## Debrief Questions (for class discussion)

1. What surprised you about the Dockerfile requirements?
2. Why is the "This repo" toggle important?
3. How would you apply this pattern to a real data/ML pipeline?